# IDA CFG Analysis with ipysigma

This notebook demonstrates how to extract, load, and visualize the Control Flow Graph (CFG) from an IDA database.


### 1. Extract CFG from IDA Database
This step requires `idapro` 9.0+ and a valid license.


In [1]:
import sys
import os
from src.extract_cfg import extract_cfg_from_db
import os

# Replace with the path to your IDA database (.idb or .i64)
db_path = "/Users/mark/windows_share/test/reorder_and_pad.exe.i64"
json_path = "reorder_and_pad2.json"

if os.path.exists(db_path):
    json_path = extract_cfg_from_db(db_path, output_path=json_path)
else:
    print(f"Database {db_path} not found. Skipping extraction.")


Opening database: /Users/mark/windows_share/test/reorder_and_pad.exe.i64
Number of functions found: 343
Extracting functions and intra-function edges...
Found 343 functions.
Extracting inter-function calls...
Found non-call link to main 5368713352 from sub_140001000 @ 0x140001000
CFG exported to reorder_and_pad2.json


### 2. Load and Visualize the Graph with ipysigma


In [6]:
from typing import Hashable
from networkx import k_core, DiGraph
import importlib
importlib.invalidate_caches()
from ipysigma import Sigma
import pandas as pd
import networkx as nx
import src.visualize_cfg as cfg
importlib.reload(cfg)
import os

def prune_graph(og: DiGraph[Hashable]) -> DiGraph[Hashable]:
    to_return = og.copy()
    print(f"Graph loaded: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    print(f"entrypoint is {entrypoint}")

    # remove self loops
    to_return.remove_edges_from(nx.selfloop_edges(to_return))

    to_return = cfg.collapse_chains(to_return)


    # remove nodes with degree <= 1
    to_return = k_core(to_return, 2)
    entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    print(f"entrypoint is {entrypoint}")
    # to_be_removed = [x for  x in G.nodes() if G.degree()[x] <= 1]
    # print(f"Number of nodes to be removed: {len(to_be_removed)}")
    # G.remove_nodes_from(to_be_removed)
    # # Basic info
    # print(f"Number of functions after removing degree <= 1: {len(G.nodes)}")
    #
    # print(f"Number of nodes after collapsing chains: {len(G.nodes)}")
    # entrypoint = [x for x in G.nodes() if G.nodes[x]['entry_point'] == True]
    # if not entrypoint:
    #     print("No entrypoint found. Please check the graph.")
    #     raise ValueError("No entrypoint found")
    print(f"Graph pruned: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    return to_return


# Load the graph data
G = cfg.load_cfg(json_path)
if G is None:
    print(f"Could not load graph from {json_path}. Please make sure the file exists and it is a valid CFG JSON.")
    exit()

assert(G is not None)
pruned = prune_graph(G)

# Visualize with ipysigma
widget = Sigma(
    pruned,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in d.get("func") else d.get('color'),
    node_size=lambda n, d: 10 if (d.get('entry_point') or ("main" in d.get('func'))) else 1,
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget)


Graph loaded: 5174 nodes, 8778 edges
entrypoint is [5368796632]
entrypoint is []
Graph pruned: 3711 nodes, 7048 edges


Sigma(nx.DiGraph with 3,711 nodes and 7,048 edges)

Ways to reduce:
- strongly connected components
- remove nodes with degree <= 1
- clique detection and collapse
- graph community collapse
min cuts
- graph spectral?
- laplacian?
-